# 20 — Player challengers (2026)

Who challenges, how often, and how well. Source: `challenges_enriched`.
Every rate carries a 95% confidence interval (Wilson). Players are listed
in full; rankings use a minimum of 15 challenges.


In [ ]:
import os
import sys

import pandas as pd

sys.path.insert(0, "../src")
from stats_helpers import rate_table, wilson_interval

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

TABLES = "../data/tables"
REPORTS = "../reports/tables"
os.makedirs(REPORTS, exist_ok=True)
MIN_N = 15

In [ ]:
chal = pd.read_parquet(f"{TABLES}/challenges_enriched/all_seasons.parquet")
print(chal.shape)
chal.head(3)

In [ ]:
won, n = int(chal["is_overturned"].sum()), len(chal)
lo, hi = wilson_interval(won, n)
LEAGUE_RATE = won / n
print(f"challenges {n:,}  overturned {won:,}  rate {LEAGUE_RATE:.3f}  CI {lo:.3f}-{hi:.3f}")
rate_table(chal, ["challenger_role"], "is_overturned")

In [ ]:
all_players = rate_table(
    chal, ["challenger_id", "challenger_name", "challenger_role"], "is_overturned"
)
all_players = all_players.rename(columns={"trials": "challenges", "successes": "won"})
all_players["lost"] = all_players["challenges"] - all_players["won"]
all_players["vs_league"] = all_players["rate"] - LEAGUE_RATE
all_players = all_players.sort_values("challenges", ascending=False).reset_index(drop=True)

print("challengers:", len(all_players))
print("challenges per player: median", all_players["challenges"].median(),
      "| max", all_players["challenges"].max())
print(all_players["challenger_role"].value_counts().to_string())
all_players.head(10)

In [ ]:
ranked = all_players[all_players["challenges"] >= MIN_N].copy()
print(f"players with >= {MIN_N} challenges: {len(ranked)}")

print("\n--- best: top 15 by lower confidence bound ---")
display(ranked.sort_values("ci_low", ascending=False).head(15))

print("\n--- worst: bottom 10 by upper confidence bound ---")
display(ranked.sort_values("ci_high").head(10))

print("\n--- busiest: top 10 by challenges ---")
display(ranked.head(10))

In [ ]:
by_role = ranked.groupby("challenger_role").agg(
    players=("challenges", "size"),
    challenges=("challenges", "sum"),
    won=("won", "sum"),
)
by_role["rate"] = by_role["won"] / by_role["challenges"]
by_role

In [ ]:
all_players.to_csv(f"{REPORTS}/player_challengers_2026_all.csv", index=False)
ranked.sort_values("ci_low", ascending=False).to_csv(
    f"{REPORTS}/player_challengers_2026_ranked.csv", index=False
)
print("saved:", len(all_players), "players;", len(ranked), "ranked")